In [3]:
from sklearn.metrics.cluster import fowlkes_mallows_score

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


X, y = make_regression(n_samples=200, noise=10, random_state=42)


X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


for degree in [1, 2,3]:
    poly = PolynomialFeatures(degree=degree,include_bias=False)
    X_train_p = poly.fit_transform(X_train)
    X_test_p = poly.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_p, y_train)

    train_mse = mean_squared_error(y_train, model.predict(X_train_p))
    test_mse = mean_squared_error(y_test, model.predict(X_test_p))

    print("degree:", degree, "train_mse:", train_mse, "test_mse:", test_mse)

degree: 1 train_mse: 36.59286153711178 test_mse: 431.3140955019114
degree: 2 train_mse: 2.0435848276070134e-25 test_mse: 31632.59759581086
degree: 3 train_mse: 2.92984328594277e-25 test_mse: 28136.041534095188



k-fold CV (k=5)

1.  split data into 5 folds
2.  repeat 5 times: train on 4, test on 1
3.  return 5 test scores



In [4]:


from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
import numpy as np

In [6]:

model = Ridge(alpha=1.0)


scores = cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=5)

In [7]:


mse_scores = -scores
print("Fold MSE:", mse_scores)
print("Mean MSE:", mse_scores.mean())
print("Std MSE:", mse_scores.std())

Fold MSE: [462.8288227  273.38281387 358.1524678  265.31478888 585.35899825]
Mean MSE: 389.00757830121495
Std MSE: 121.34618773554041




Ridge vs Lasso

*  Ridge (L2): shrink coefficients smoothly
*  Lasso (L1): can shrink some coefficients to zero
* alpha: penalty strength (controls complexity)



In [9]:

from sklearn.linear_model import Ridge, Lasso
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.01)

In [10]:


import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [12]:
rng = np.random.default_rng(42)
B = 200
oob_mse = []


n = len(X)
idx_all = np.arange(n)

In [13]:

for b in range(B):
    idx_boot = rng.integers(0, n, n)  # sample n indices WITH replacement
    idx_oob = np.setdiff1d(idx_all, np.unique(idx_boot))  # points never selected
    if len(idx_oob) == 0:
        continue

In [15]:

    X_boot, y_boot = X[idx_boot], y[idx_boot]
    X_oob, y_oob = X[idx_oob], y[idx_oob]

    model = Ridge(alpha=1.0)
    model.fit(X_boot, y_boot)
    preds = model.predict(X_oob)
    oob_mse.append(mean_squared_error(y_oob, preds))

In [16]:


print("Bootstrap OOB MSE mean:", np.mean(oob_mse))
print("Bootstrap OOB MSE std:", np.std(oob_mse))

Bootstrap OOB MSE mean: 1662.850501868881
Bootstrap OOB MSE std: 0.0


In [18]:

from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.linear_model import Ridge
import numpy as np

param_grid = {"alpha": [0.01, 0.1, 1, 10, 100]}


inner = KFold(n_splits=5, shuffle=True, random_state=42)
outer = KFold(n_splits=5, shuffle=True, random_state=123)

In [19]:
ridge = Ridge()
search = GridSearchCV(
    estimator=ridge,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=inner
)



In [20]:

nested_scores = cross_val_score(search, X, y, scoring="neg_mean_squared_error", cv=outer)

In [21]:

nested_mse = -nested_scores
print("Nested CV MSE:", nested_mse)
print("Mean:", nested_mse.mean(), "Std:", nested_mse.std())

Nested CV MSE: [396.90488375 395.59138826 400.05403372 364.1325383  506.6121085 ]
Mean: 412.6589905047831 Std: 48.74501986172402


Fairness check: error rates by group, not just overall accuracy


Prediction ≠ Decision Threshold choice changes who is helped/harmed

In [24]:

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

Xc, yc = make_classification(
    n_samples=3000,
    n_features=10,
    n_informative=5,
    weights=[0.7, 0.3],
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(Xc, yc, test_size=0.3, random_state=42)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
print(proba)



[0.42438201 0.28117741 0.05976897 0.34184574 0.15578501 0.22240686
 0.31816562 0.43571218 0.99068085 0.15438604 0.8550387  0.21434559
 0.05119453 0.02572381 0.10136111 0.28325257 0.24681566 0.63526733
 0.2552913  0.14988321 0.3809299  0.12889977 0.90392133 0.56322635
 0.05795685 0.20102875 0.11463508 0.62241548 0.89885077 0.18615726
 0.96644104 0.72819119 0.41946679 0.18861359 0.39130378 0.2332622
 0.28748021 0.48122792 0.89870706 0.51840537 0.12238662 0.26231923
 0.56945508 0.41734375 0.28706864 0.07185904 0.11459117 0.28345761
 0.45420701 0.32187957 0.09259053 0.03064249 0.22764299 0.04888049
 0.55833498 0.18103022 0.09862119 0.73879766 0.0881738  0.28129959
 0.08421523 0.34085463 0.13275583 0.19343323 0.1968412  0.03721837
 0.47652945 0.95868109 0.21919621 0.22163976 0.22030927 0.14240376
 0.78807895 0.22296437 0.03266848 0.18985178 0.21087642 0.1377517
 0.14864344 0.4549419  0.04674467 0.11767875 0.18457301 0.3334393
 0.30765848 0.21687927 0.02168822 0.18416065 0.17941754 0.4314672

In [25]:
rng = np.random.default_rng(42)
group = rng.choice(["A", "B"], size=len(y_test))


In [26]:

def rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn, "FPR": fpr, "FNR": fnr}

In [27]:

thresholds = [0.3, 0.5, 0.7]
rows = []

for t in thresholds:
    y_pred = (proba >= t).astype(int)
    for g in ["A", "B"]:
        mask = (group == g)
        r = rates(y_test[mask], y_pred[mask])
        r.update({"threshold": t, "group": g})
        rows.append(r)

results = pd.DataFrame(rows)
print(results)

    TP  FP   TN   FN       FPR       FNR  threshold group
0  101  69  232   37  0.229236  0.268116        0.3     A
1   97  85  234   45  0.266458  0.316901        0.3     B
2   73   8  293   65  0.026578  0.471014        0.5     A
3   69  16  303   73  0.050157  0.514085        0.5     B
4   46   3  298   92  0.009967  0.666667        0.7     A
5   39   1  318  103  0.003135  0.725352        0.7     B


In [28]:

from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10)
for mp, fp in zip(mean_pred, frac_pos):
    print("mean_pred:", round(mp, 3), "frac_pos:", round(fp, 3))

mean_pred: 0.055 frac_pos: 0.128
mean_pred: 0.151 frac_pos: 0.128
mean_pred: 0.248 frac_pos: 0.199
mean_pred: 0.343 frac_pos: 0.242
mean_pred: 0.443 frac_pos: 0.368
mean_pred: 0.556 frac_pos: 0.733
mean_pred: 0.639 frac_pos: 0.75
mean_pred: 0.746 frac_pos: 0.862
mean_pred: 0.862 frac_pos: 1.0
mean_pred: 0.939 frac_pos: 1.0


In [ ]:

import sys, numpy as np, sklearn, pandas as pd
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)

In [29]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

acc = cross_val_score(pipe, Xc, yc, cv=5, scoring="accuracy")
print("CV accuracy mean/std:", acc.mean(), acc.std())

CV accuracy mean/std: 0.8119999999999999 0.010349449797506665



	1	Model overview + intended use
	2	Data summary + provenance
	3	Training procedure + preprocessing
	4	Evaluation method (ESL Ch.7) + uncertainty
	5	Fairness checks + subgroup metrics
	6	Calibration (if probabilities used)
	7	Limitations + failure modes
	8	Ethical risks + mitigations
	9	Reproducibility (versions, seeds, repo)

“We evaluated models using 5-fold cross-validation to estimate expected test error. Reported performance includes mean and standard deviation across folds to reflect uncertainty.”

“We compared multiple models; differences in mean performance were small relative to fold-to-fold variability, so we prioritized interpretability and stability.”

“We examined subgroup error rates across decision thresholds to identify disparities in false positive and false negative rates.”

“This model is intended for decision support, not fully automated decisions. Performance may degrade under dataset shift. We recommend periodic monitoring and re-validation.”

cross_val_score takes the model, splits the data into folds, repeatedly calls .fit() on the training folds, then calls .predict() on the held-out fold, computes a score, and repeats until each fold has been held out once. The returned array contains one score per fold. Mean estimates expected test error; standard deviation quantifies instability due to sampling.


Ridge uses an L2 penalty that smoothly shrinks coefficients; it usually keeps all features but reduces variance. Lasso uses an L1 penalty that can set coefficients exactly to zero, acting like feature selection. alpha is the penalty strength: larger alpha yields simpler, more stable models; smaller alpha yields more flexible, higher-variance models. Hyperparameters should be tuned with care, ideally with nested CV.